In [ ]:
def morphology_preprocess(img):
    img = img.astype(np.uint8)

    # Resize
    img = cv2.resize(img, (224, 224))

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Otsu Threshold
    _, binary = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Structuring element
    kernel = np.ones((5, 5), np.uint8)

    # Opening
    opened = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel
    )

    # Closing
    closed = cv2.morphologyEx(
        opened,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Apply mask to original image
    segmented = cv2.bitwise_and(
        img,
        img,
        mask=closed
    )

    # MobileNetV2 preprocessing
    segmented = preprocess_input(
        segmented.astype(np.float32)
    )

    return segmented

In [ ]:
sample_class = classes[0]

sample_path = os.path.join(
    dataset_dir,
    sample_class,
    os.listdir(os.path.join(dataset_dir, sample_class))[0]
)

img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (224, 224))

gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

_, binary = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

kernel = np.ones((5,5), np.uint8)

opened = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    kernel
)

closed = cv2.morphologyEx(
    opened,
    cv2.MORPH_CLOSE,
    kernel
)

segmented = cv2.bitwise_and(
    img,
    img,
    mask=closed
)

plt.figure(figsize=(16,5))

plt.subplot(1,4,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(binary, cmap="gray")
plt.title("Threshold")

plt.subplot(1,4,3)
plt.imshow(opened, cmap="gray")
plt.title("Opening")

plt.subplot(1,4,4)
plt.imshow(segmented)
plt.title("Opening + Closing")
plt.axis("off")

plt.show()

In [ ]:
morph_datagen = ImageDataGenerator(
    preprocessing_function=morphology_preprocess,
    validation_split=0.2
)

morph_train = morph_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

morph_val = morph_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)